**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cpu


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [2]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

100%|██████████| 9.91M/9.91M [00:00<00:00, 58.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.79MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.96MB/s]


torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [3]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [4]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [5]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [6]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [7]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    # Returns (center, tau).
    # TODO: tau = lam * eps; PGD center in the (eps - tau)-ball around x
    tau = lam * eps
    model.eval()
    center = pgd(model, x, y, eps - tau, norm, steps=5)   # the small region stays inside the eps-ball
    model.train()
    return center, tau

def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    # z_hat = -(IBP lower bounds of the margins f_y - f_j) on the small region. z_hat[y] = 0.
    center, tau = propagation_region(model, x, y, eps, lam, norm)
    z_hat = -ibp_margin_bounds(model, center, tau, norm, y)
    return F.cross_entropy(z_hat, y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    # TODO: standard loss for t < 1, then the certified loss with the scheduled radius
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)

In [8]:
models, results = {}, []
for name, loss_fn in [("standard", standard_loss),
                      ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
                      ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]:
    models[name] = make_model()
    sec = train(models[name], loss_fn)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5638  time 1 s
epoch  2  loss 0.1993  time 1 s
epoch  3  loss 0.1511  time 2 s
epoch  4  loss 0.1202  time 2 s
epoch  5  loss 0.1016  time 3 s
epoch  6  loss 0.0863  time 3 s
epoch  7  loss 0.0770  time 4 s
epoch  8  loss 0.0666  time 4 s
epoch  9  loss 0.0597  time 5 s
epoch 10  loss 0.0521  time 5 s
epoch 11  loss 0.0478  time 6 s
epoch 12  loss 0.0440  time 7 s
epoch 13  loss 0.0393  time 7 s
epoch 14  loss 0.0369  time 8 s
epoch 15  loss 0.0322  time 8 s
epoch 16  loss 0.0156  time 9 s
epoch 17  loss 0.0119  time 9 s
epoch 18  loss 0.0109  time 10 s
epoch 19  loss 0.0102  time 10 s
epoch 20  loss 0.0094  time 11 s
epoch  1  loss 0.5674  time 0 s
epoch  2  loss 0.5071  time 4 s
epoch  3  loss 0.4931  time 7 s
epoch  4  loss 0.5037  time 10 s
epoch  5  loss 0.5218  time 14 s
epoch  6  loss 0.5626  time 17 s
epoch  7  loss 0.6168  time 20 s
epoch  8  loss 0.6749  time 24 s
epoch  9  loss 0.7315  time 27 s
epoch 10  loss 0.7869  time 30 s
epoch 11  loss 0.8460  time 34

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [9]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)

    center, tau = propagation_region(model, x, y, eps_scale(t) * EPS["linf"], LAM["linf"], "linf")
    z_linf = -ibp_margin_bounds(model, center, tau, "linf", y)
    center, tau = propagation_region(model, x, y, eps_scale(t) * EPS["l2"], LAM["l2"], "l2")
    z_l2 = -ibp_margin_bounds(model, center, tau, "l2", y)

    loss_linf = F.cross_entropy(z_linf, y, reduction="none")
    loss_l2 = F.cross_entropy(z_l2, y, reduction="none")
    loss = torch.maximum(loss_linf, loss_l2).mean()

    if eta > 0:
        gamma = (z_l2 >= 0).sum(dim=1) == 1
        if gamma.sum() > 0:
            log_d2 = F.log_softmax(z_l2[gamma], dim=1)
            log_dinf = F.log_softmax(z_linf[gamma], dim=1)
            kl = (log_d2.exp() * (log_d2 - log_dinf)).sum(dim=1).mean()
            loss = loss + eta * kl
    return loss


def gradient_projection(g_n, g_c, beta):
    cos = F.cosine_similarity(g_n.flatten(), g_c.flatten(), dim=0)
    if cos > 0:
        gp = cos * g_n
    else:
        gp = torch.zeros_like(g_n)
    return beta * gp + (1 - beta) * g_c

In [12]:
def mean_results(run0, run1, name):
    mean = {"name": name}
    for k in run0:
        if k != "name":
            mean[k] = (run0[k] + run1[k]) / 2
    return mean


for name, loss_fn, beta in [("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
                            ("cure_max_ba", cure_loss, None),
                            ("cure_max_ba_gp", cure_loss, 0.5)]:
    runs = []
    for seed in [0, 1]:
        torch.manual_seed(seed)
        run_name = name + "_s" + str(seed)
        models[run_name] = make_model()
        sec = train(models[run_name], loss_fn, seed=seed, gp_beta=beta)
        run = dict(evaluate(models[run_name], run_name, attacks=True), time=sec)
        print_results([run])
        runs.append(run)
    results.append(mean_results(runs[0], runs[1], name))
print_results(results)

epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.4906  time 7 s
epoch  3  loss 0.5016  time 14 s
epoch  4  loss 0.5312  time 21 s
epoch  5  loss 0.5948  time 27 s
epoch  6  loss 0.6653  time 34 s
epoch  7  loss 0.7542  time 41 s
epoch  8  loss 0.8595  time 47 s
epoch  9  loss 0.9762  time 54 s
epoch 10  loss 1.1098  time 61 s
epoch 11  loss 1.2511  time 67 s
epoch 12  loss 1.3038  time 74 s
epoch 13  loss 1.2821  time 81 s
epoch 14  loss 1.2615  time 87 s
epoch 15  loss 1.2468  time 94 s
epoch 16  loss 1.1951  time 101 s
epoch 17  loss 1.1890  time 107 s
epoch 18  loss 1.1814  time 114 s
epoch 19  loss 1.1788  time 120 s
epoch 20  loss 1.1763  time 127 s
cure_max_s0               clean  92.54  linf  67.39  l2  56.50  union  56.37  pgd_linf  76.68  pgd_l2  78.08  pgd_union  76.07  time/epoch   6.3 s
epoch  1  loss 0.5738  time 0 s
epoch  2  loss 0.4911  time 7 s
epoch  3  loss 0.5046  time 14 s
epoch  4  loss 0.5431  time 21 s
epoch  5  loss 0.5964  time 27 s
epoch  6  loss 0.6645  time

# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

In [13]:
def ablation_all_images_loss(model, x, y, t):
    loss = cure_loss(model, x, y, t)
    if t < WARMUP:
        return loss
    eps = eps_scale(t) * EPS["l2"]
    z_full = -ibp_margin_bounds(model, x, eps, "l2", y)
    return loss + 0.5 * F.cross_entropy(z_full, y)

*In the output below, my_method is the ablation (full-ball loss for all images). My final method is my_method_on_certified_images.*

In [14]:
for r in results:
    if r["name"] == "cure_max_ba":
        best_b = r

runs = []
for seed in [0, 1]:
    torch.manual_seed(seed)
    run_name = "my_method_s" + str(seed)
    models[run_name] = make_model()
    sec = train(models[run_name], ablation_all_images_loss, seed=seed)
    run = dict(evaluate(models[run_name], run_name, attacks=True), time=sec)
    print_results([run])
    runs.append(run)
my_result = mean_results(runs[0], runs[1], "my_method")

ratio = my_result["time"] / best_b["time"]
epochs_same_time = round(20 * ratio)
print("my method uses", round(ratio, 2), "x the time per epoch of CURE-Max + BA")
print("CURE-Max + BA with the same total time:", epochs_same_time, "epochs")

runs = []
for seed in [0, 1]:
    torch.manual_seed(seed)
    run_name = "cure_max_ba_" + str(epochs_same_time) + "ep_s" + str(seed)
    models[run_name] = make_model()
    sec = train(models[run_name], cure_loss, epochs=epochs_same_time, seed=seed)
    run = dict(evaluate(models[run_name], run_name, attacks=True), time=sec)
    print_results([run])
    runs.append(run)
same_time_result = mean_results(runs[0], runs[1], "cure_max_ba_" + str(epochs_same_time) + "ep")

print_results([best_b, my_result, same_time_result])

epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.7130  time 8 s
epoch  3  loss 0.7342  time 15 s
epoch  4  loss 0.7958  time 22 s
epoch  5  loss 0.9019  time 30 s
epoch  6  loss 1.0231  time 37 s
epoch  7  loss 1.1855  time 44 s
epoch  8  loss 1.3686  time 51 s
epoch  9  loss 1.5706  time 59 s
epoch 10  loss 1.7936  time 66 s
epoch 11  loss 2.0173  time 73 s
epoch 12  loss 2.0991  time 81 s
epoch 13  loss 2.0696  time 88 s
epoch 14  loss 2.0397  time 95 s
epoch 15  loss 2.0203  time 102 s
epoch 16  loss 1.9553  time 109 s
epoch 17  loss 1.9482  time 117 s
epoch 18  loss 1.9385  time 124 s
epoch 19  loss 1.9362  time 131 s
epoch 20  loss 1.9335  time 138 s
my_method_s0              clean  91.41  linf  66.37  l2  59.39  union  59.04  pgd_linf  74.98  pgd_l2  76.69  pgd_union  74.22  time/epoch   6.9 s
epoch  1  loss 0.5738  time 0 s
epoch  2  loss 0.7140  time 7 s
epoch  3  loss 0.7400  time 15 s
epoch  4  loss 0.8068  time 22 s
epoch  5  loss 0.9149  time 29 s
epoch  6  loss 1.0494  tim

In [15]:
def linf_full_ball_loss(model, x, y, t):
    loss = cure_loss(model, x, y, t)
    if t < WARMUP:
        return loss
    eps = eps_scale(t) * EPS["linf"]
    z_full = -ibp_margin_bounds(model, x, eps, "linf", y)
    return loss + 0.5 * F.cross_entropy(z_full, y)


runs = []
for seed in [0, 1]:
    torch.manual_seed(seed)
    run_name = "ablation_full_ball_linf_s" + str(seed)
    models[run_name] = make_model()
    sec = train(models[run_name], linf_full_ball_loss, seed=seed)
    run = evaluate(models[run_name], run_name, attacks=True)
    run["time"] = sec
    print_results([run])
    runs.append(run)
ablation_result = mean_results(runs[0], runs[1], "ablation_full_ball_linf")

print_results(results + [my_result, same_time_result, ablation_result])

epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.7098  time 8 s
epoch  3  loss 0.7138  time 16 s
epoch  4  loss 0.7623  time 24 s
epoch  5  loss 0.8601  time 32 s
epoch  6  loss 0.9633  time 39 s
epoch  7  loss 1.1014  time 47 s
epoch  8  loss 1.2640  time 55 s
epoch  9  loss 1.4445  time 62 s
epoch 10  loss 1.6440  time 70 s
epoch 11  loss 1.8496  time 77 s
epoch 12  loss 1.9233  time 85 s
epoch 13  loss 1.8889  time 92 s
epoch 14  loss 1.8583  time 100 s
epoch 15  loss 1.8384  time 108 s
epoch 16  loss 1.7617  time 115 s
epoch 17  loss 1.7503  time 123 s
epoch 18  loss 1.7393  time 131 s
epoch 19  loss 1.7370  time 139 s
epoch 20  loss 1.7335  time 147 s
ablation_full_ball_linf_s0clean  92.65  linf  71.41  l2  56.14  union  56.11  pgd_linf  78.18  pgd_l2  78.68  pgd_union  76.99  time/epoch   7.3 s
epoch  1  loss 0.5738  time 0 s
epoch  2  loss 0.7118  time 8 s
epoch  3  loss 0.7222  time 15 s
epoch  4  loss 0.7786  time 23 s
epoch  5  loss 0.8598  time 30 s
epoch  6  loss 0.9635  ti

In [16]:
for r in [best_b, my_result, same_time_result, ablation_result]:
    print(r["name"])
    print("linf: certified", round(r["linf"], 2), " PGD", round(r["pgd_linf"], 2),
          " gap", round(r["pgd_linf"] - r["linf"], 2))
    print("l2:   certified", round(r["l2"], 2), " PGD", round(r["pgd_l2"], 2),
          " gap", round(r["pgd_l2"] - r["l2"], 2))

cure_max_ba
linf: certified 66.97  PGD 76.43  gap 9.46
l2:   certified 57.08  PGD 77.86  gap 20.78
my_method
linf: certified 66.65  PGD 75.4  gap 8.75
l2:   certified 60.23  PGD 77.4  gap 17.17
cure_max_ba_22ep
linf: certified 67.16  PGD 76.42  gap 9.27
l2:   certified 57.09  PGD 77.99  gap 20.91
ablation_full_ball_linf
linf: certified 71.52  PGD 78.8  gap 7.29
l2:   certified 56.17  PGD 79.32  gap 23.15


In [17]:
def my_loss(model, x, y, t):
    if t < WARMUP:
        return standard_loss(model, x, y, t)

    center, tau = propagation_region(model, x, y, eps_scale(t) * EPS["linf"], LAM["linf"], "linf")
    z_linf = -ibp_margin_bounds(model, center, tau, "linf", y)
    center, tau = propagation_region(model, x, y, eps_scale(t) * EPS["l2"], LAM["l2"], "l2")
    z_l2 = -ibp_margin_bounds(model, center, tau, "l2", y)

    loss_linf = F.cross_entropy(z_linf, y, reduction="none")
    loss_l2 = F.cross_entropy(z_l2, y, reduction="none")
    loss = torch.maximum(loss_linf, loss_l2).mean()

    gamma = (z_l2 >= 0).sum(dim=1) == 1
    if gamma.sum() > 0:
        log_d2 = F.log_softmax(z_l2[gamma], dim=1)
        log_dinf = F.log_softmax(z_linf[gamma], dim=1)
        kl = (log_d2.exp() * (log_d2 - log_dinf)).sum(dim=1).mean()
        loss = loss + 0.5 * kl
        eps2 = eps_scale(t) * EPS["l2"]
        z_full = -ibp_margin_bounds(model, x[gamma], eps2, "l2", y[gamma])
        loss = loss + 0.5 * F.cross_entropy(z_full, y[gamma])
    return loss

In [18]:
runs = []
for seed in [0, 1]:
    torch.manual_seed(seed)
    run_name = "my_method_on_certified_images_s" + str(seed)
    models[run_name] = make_model()
    sec = train(models[run_name], my_loss, seed=seed)
    run = dict(evaluate(models[run_name], run_name, attacks=True), time=sec)
    print_results([run])
    runs.append(run)
focus_result = mean_results(runs[0], runs[1], "my_method_on_certified_images")

print_results([best_b, my_result, focus_result])

epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.5479  time 8 s
epoch  3  loss 0.5575  time 15 s
epoch  4  loss 0.6020  time 23 s
epoch  5  loss 0.6848  time 30 s
epoch  6  loss 0.7807  time 38 s
epoch  7  loss 0.8962  time 45 s
epoch  8  loss 1.0361  time 52 s
epoch  9  loss 1.1954  time 60 s
epoch 10  loss 1.3792  time 67 s
epoch 11  loss 1.5653  time 74 s
epoch 12  loss 1.6318  time 81 s
epoch 13  loss 1.6011  time 89 s
epoch 14  loss 1.5736  time 96 s
epoch 15  loss 1.5543  time 103 s
epoch 16  loss 1.4930  time 110 s
epoch 17  loss 1.4837  time 117 s
epoch 18  loss 1.4755  time 125 s
epoch 19  loss 1.4723  time 132 s
epoch 20  loss 1.4700  time 139 s
my_method_on_certified_images_s0clean  91.77  linf  67.92  l2  61.27  union  60.95  pgd_linf  75.59  pgd_l2  77.17  pgd_union  75.08  time/epoch   7.0 s
epoch  1  loss 0.5738  time 0 s
epoch  2  loss 0.5507  time 7 s
epoch  3  loss 0.5613  time 15 s
epoch  4  loss 0.6133  time 22 s
epoch  5  loss 0.6888  time 29 s
epoch  6  loss 0.781